## THIS NOTEBOOK DOES PREPROCESSING OF THE DATASET

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import joblib
from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv('../data/raw/Customertravel.csv')
df.head()

,Age,FrequentFlyer,AnnualIncomeClass,ServicesOpted,AccountSyncedToSocialMedia,BookedHotelOrNot,Target
0,34,No,Middle Income,6,No,Yes,0
1,34,Yes,Low Income,5,Yes,No,1
2,37,No,Middle Income,3,Yes,No,0
3,30,No,Middle Income,2,No,No,0
4,30,No,Low Income,1,No,No,0


## Encoding categorical values
**One category** was dropped per column to avoid **multicolinearity**<br>
0 : False | 1 : True

In [4]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df = df.astype({col: int for col in df.select_dtypes(bool).columns})
df.head()

,Age,ServicesOpted,Target,FrequentFlyer_No Record,FrequentFlyer_Yes,AnnualIncomeClass_Low Income,AnnualIncomeClass_Middle Income,AccountSyncedToSocialMedia_Yes,BookedHotelOrNot_Yes
0,34,6,0,0,0,0,1,0,1
1,34,5,1,0,1,1,0,1,0
2,37,3,0,0,0,0,1,1,0
3,30,2,0,0,0,0,1,0,0
4,30,1,0,0,0,1,0,0,0


## Splitting the dataset
**X_train** is the training dataset<br>
**X_val** will be used for validation (important for hypertuning of parameters)<br>
**X_test** is the testing dataset<br>
Similarly Y (target) is divided.<br>

In [5]:
X = df.drop('Target', axis=1)
y = df['Target']

In [6]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

Train: (667, 8), Val: (143, 8), Test: (144, 8)


## SMOTE
As class imbalance was noticed during **EDA**, I will apply SMOTE on the training data

In [11]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'Before SMOTE: {y_train.value_counts().to_dict()}')
print(f'After SMOTE: {pd.Series(y_train_resampled).value_counts().to_dict()}')

Before SMOTE: {0: 511, 1: 156}
After SMOTE: {0: 511, 1: 511}


Smote creates (synthetic) datapoints for the minority class (in this case Churn) so the model is trained on balanced dataset.

**Scaled the numeric values of the dataset after applying SMOTE**

In [12]:
scaler = StandardScaler()

X_train_resampled[['Age']] = scaler.fit_transform(X_train_resampled[['Age']])
X_val[['Age']] = scaler.transform(X_val[['Age']])
X_test[['Age']] = scaler.transform(X_test[['Age']])

joblib.dump(scaler, '../outputs/models/scaler.pkl')

['../outputs/models/scaler.pkl']

In [13]:
X_train_resampled.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)

pd.Series(y_train_resampled).to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('All data splits are saved.')

All data splits are saved.
